In [1]:
"""Figures 2-4 of the CSJ paper.

Reads csj_out_numpy/runs.jsonl and the energy histories in csj_out_numpy/hist
and writes PNG and PDF files to csj_out_numpy/figs.

    python csj_figs.py
    python csj_figs.py --out PATH
"""
import argparse
import json
import os

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import transforms
from matplotlib.patches import Patch

E_ED = -2.224566   # MeV
TOL = 0.01         # MeV
BMAX = 2000
VAR_TOL = 1e-3     # var(Np) and var(Nn) below this: pure sector

RANGES = ["narrow", "near0", "pm_pi", "zero_2pi"]
LABELS = {
    "narrow": r"$[0.1,\ 0.5]$",
    "near0": r"$[-0.1,\ 0.1]$",
    "pm_pi": r"$[-\pi,\ \pi]$",
    "zero_2pi": r"$[0,\ 2\pi]$",
}
OPTS = ["COBYLA", "Nelder-Mead", "Powell", "SLSQP", "L-BFGS-B"]
ALL_OPTS = OPTS + ["SPSA"]
STYLE = {
    "COBYLA":      dict(color="#1f77b4", ls="-",               marker="o"),
    "Nelder-Mead": dict(color="#d62728", ls="--",              marker="s"),
    "Powell":      dict(color="#2ca02c", ls="-.",              marker="^"),
    "SLSQP":       dict(color="#9467bd", ls=":",               marker="D"),
    "L-BFGS-B":    dict(color="#ff7f0e", ls=(0, (5, 1)),       marker="v"),
    "SPSA":        dict(color="#555555", ls=(0, (3, 1, 1, 1)), marker="x"),
}
CATS = [
    ("conv",   "Converged",                    "#3b7d3b", ""),
    ("s21",    "Sector (2p,1n)",               "#c44e52", ""),
    ("s01",    "Sector (0p,1n)",               "#dd8452", ""),
    ("sother", "Other pure sector",            "#8172b3", ""),
    ("mixed",  "Multi-sector / not converged", "#cccccc", "///"),
]

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "mathtext.fontset": "stix",
    "axes.linewidth": 0.8,
    "legend.frameon": False,
})


def load(out):
    with open(os.path.join(out, "runs.jsonl"), encoding="utf-8") as f:
        runs = [json.loads(line) for line in f if line.strip()]
    print(f"{len(runs)} runs in runs.jsonl")
    return [r for r in runs if r.get("error") is None]


def protocol_of(opt):
    return "fixed" if opt == "SPSA" else "free"


def select(runs, opt, rg, depth=2):
    sel = [r for r in runs if r["depth"] == depth and r["opt"] == opt
           and r["init"] == rg and r["protocol"] == protocol_of(opt)]
    return sorted(sel, key=lambda r: r["seed"])


def first_hit(h):
    # evaluation number (from 1) at which |E - E_ED| < TOL first holds
    idx = np.flatnonzero(np.abs(h - E_ED) < TOL)
    return float(idx[0] + 1) if idx.size else np.inf


def add_hits(runs, out):
    bad = 0
    for r in runs:
        if r["depth"] != 2:
            continue
        h = np.asarray(np.load(os.path.join(out, "hist", r["hist_file"])), dtype=float).ravel()
        r["hit"] = first_hit(h)
        n = r.get("evals_to_10keV")
        if r["protocol"] == "free" and n is not None and np.isfinite(r["hit"]) \
                and int(r["hit"]) != int(n):
            bad += 1
    print(f"first hits that differ from evals_to_10keV: {bad}")


def check_counts(runs):
    bad = [f"{opt}/{rg}: {len(select(runs, opt, rg))}"
           for opt in ALL_OPTS for rg in RANGES if len(select(runs, opt, rg)) != 20]
    print("runs per cell: " + ("20 everywhere" if not bad else ", ".join(bad)))


def converged(r):
    # same criterion as in the paper; r["success"] is SciPy's flag and is not used
    return r["dE"] < TOL and r["w_phys"] > 0.99


def classify(r):
    if converged(r):
        return "conv"
    if r["var_Np"] < VAR_TOL and r["var_Nn"] < VAR_TOL:
        s = (round(r["Np"]), round(r["Nn"]))
        if s == (2, 1):
            return "s21"
        if s == (0, 1):
            return "s01"
        return "sother"
    return "mixed"


def savefig(fig, figdir, name):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(figdir, f"{name}.{ext}"), dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}.png and {name}.pdf")


def fig2(runs, figdir):
    fig, ax = plt.subplots(figsize=(7.0, 3.6))
    trans = transforms.blended_transform_factory(ax.transData, ax.transAxes)
    rng = np.random.default_rng(0)
    w = 0.16
    for j, opt in enumerate(OPTS):
        for i, rg in enumerate(RANGES):
            vals = [r["evals_to_10keV"] for r in select(runs, opt, rg)
                    if converged(r) and r.get("evals_to_10keV") is not None]
            x = i + (j - (len(OPTS) - 1) / 2) * w
            ax.text(x, 1.01, str(len(vals)), transform=trans,
                    ha="center", va="bottom", fontsize=6.5)
            if not vals:
                continue
            bp = ax.boxplot([vals], positions=[x], widths=w * 0.8, whis=(0, 100),
                            patch_artist=True, showfliers=False,
                            medianprops=dict(color="black", lw=1.2))
            for b in bp["boxes"]:
                b.set_facecolor(STYLE[opt]["color"])
                b.set_alpha(0.55)
            jit = rng.uniform(-w * 0.22, w * 0.22, len(vals))
            ax.scatter(x + jit, vals, s=5, color="black", lw=0, zorder=3)
    ax.set_yscale("log")
    ax.set_xticks(range(len(RANGES)))
    ax.set_xticklabels([LABELS[rg] for rg in RANGES])
    ax.set_xlim(-0.55, len(RANGES) - 0.45)
    ax.set_xlabel("Initialization range (rad)")
    ax.set_ylabel(r"Evaluations to reach $|\Delta E| < 10$ keV")
    ax.grid(axis="y", which="major", lw=0.4, alpha=0.5)
    ax.text(-0.01, 1.01, "$n$ =", transform=ax.transAxes, ha="right", va="bottom",
            fontsize=6.5)
    handles = [Patch(facecolor=STYLE[o]["color"], alpha=0.55, label=o) for o in OPTS]
    fig.legend(handles=handles, loc="upper center", ncol=5, bbox_to_anchor=(0.5, 0.0))
    savefig(fig, figdir, "fig2_evals_to_target")


def fig3(runs, figdir):
    B = np.arange(1, BMAX + 1)
    fig, axes = plt.subplots(2, 2, figsize=(7.0, 5.2), sharex=True, sharey=True)
    print("\nruns within 10 keV at B = 500, 1000, 2000 (out of 20)")
    for ax, rg in zip(axes.ravel(), RANGES):
        print(f"  {rg}")
        for opt in ALL_OPTS:
            hits = np.array([r["hit"] for r in select(runs, opt, rg)])
            if hits.size == 0:
                continue
            frac = np.mean(hits[:, None] <= B[None, :], axis=0)
            st = STYLE[opt]
            ax.step(B, 100 * frac, where="post", color=st["color"], ls=st["ls"],
                    lw=1.3, label=opt)
            ax.plot(B[249::250], 100 * frac[249::250], ls="none", marker=st["marker"],
                    ms=3.5, color=st["color"])
            c = [int(np.sum(hits <= b)) for b in (500, 1000, 2000)]
            print(f"    {opt:12s} {c[0]:3d} {c[1]:5d} {c[2]:5d}")
        ax.set_title(LABELS[rg], fontsize=9)
        ax.set_xlim(0, BMAX)
        ax.set_ylim(-3, 103)
        ax.grid(lw=0.4, alpha=0.5)
    for ax in axes[1]:
        ax.set_xlabel("Evaluation budget $B$")
    for ax in axes[:, 0]:
        ax.set_ylabel(r"Runs within 10 keV (%)")
    handles = [plt.Line2D([], [], color=STYLE[o]["color"], ls=STYLE[o]["ls"],
                          marker=STYLE[o]["marker"], ms=3.5, lw=1.3, label=o)
               for o in ALL_OPTS]
    fig.legend(handles=handles, loc="lower center", ncol=6, bbox_to_anchor=(0.5, -0.03))
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    savefig(fig, figdir, "fig3_success_vs_budget")


def fig4(runs, figdir):
    fig, axes = plt.subplots(1, 4, figsize=(7.2, 3.2), sharey=True)
    print("\nfinal states (conv, s21, s01, sother, mixed)")
    used = {}
    for ax, rg in zip(axes, RANGES):
        print(f"  {rg}")
        bottoms = np.zeros(len(ALL_OPTS))
        counts = {k: np.zeros(len(ALL_OPTS)) for k, *_ in CATS}
        for i, opt in enumerate(ALL_OPTS):
            for r in select(runs, opt, rg):
                counts[classify(r)][i] += 1
            print(f"    {opt:12s} " + " ".join(f"{int(counts[k][i]):3d}" for k, *_ in CATS))
        for k, label, color, hatch in CATS:
            ax.bar(range(len(ALL_OPTS)), counts[k], bottom=bottoms, color=color,
                   hatch=hatch, edgecolor="black", lw=0.4, width=0.75)
            bottoms += counts[k]
            used[k] = used.get(k, 0) + counts[k].sum()
        # SPSA is run with a fixed budget, so it is set apart by a dashed line
        ax.axvline(len(OPTS) - 0.5, color="black", ls="--", lw=0.6)
        ax.set_xticks(range(len(ALL_OPTS)))
        ax.set_xticklabels(ALL_OPTS, rotation=60, ha="right", fontsize=7.5)
        ax.set_title(LABELS[rg], fontsize=9)
        ax.set_ylim(0, 20.5)
    axes[0].set_ylabel("Number of runs (out of 20)")
    handles = [Patch(facecolor=c, hatch=h, edgecolor="black", lw=0.4, label=l)
               for k, l, c, h in CATS if used.get(k, 0) > 0]
    fig.legend(handles=handles, loc="lower center", ncol=5, bbox_to_anchor=(0.5, -0.12),
               fontsize=8)
    fig.tight_layout()
    savefig(fig, figdir, "fig4_final_states")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default="csj_out_numpy")
    args, _ = ap.parse_known_args()
    out = args.out
    if not os.path.isfile(os.path.join(out, "runs.jsonl")):
        raise SystemExit(f"runs.jsonl not found in {out}, give the folder with --out PATH")
    figdir = os.path.join(out, "figs")
    os.makedirs(figdir, exist_ok=True)

    runs = load(out)
    check_counts(runs)
    add_hits(runs, out)
    fig2(runs, figdir)
    fig3(runs, figdir)
    fig4(runs, figdir)


if __name__ == "__main__":
    main()

680 runs in runs.jsonl
runs per cell: 20 everywhere
first hits that differ from evals_to_10keV: 0
saved fig2_evals_to_target.png and fig2_evals_to_target.pdf

runs within 10 keV at B = 500, 1000, 2000 (out of 20)
  narrow
    COBYLA        20    20    20
    Nelder-Mead    0     1     4
    Powell         5    19    20
    SLSQP         20    20    20
    L-BFGS-B      20    20    20
    SPSA          15    19    19
  near0
    COBYLA        18    19    20
    Nelder-Mead    0     0     0
    Powell         5    12    12
    SLSQP         17    19    19
    L-BFGS-B      13    20    20
    SPSA           0     0     2
  pm_pi
    COBYLA        14    15    16
    Nelder-Mead    1     2     8
    Powell         1    11    14
    SLSQP         13    14    14
    L-BFGS-B      13    16    17
    SPSA           4    10    13
  zero_2pi
    COBYLA        16    16    16
    Nelder-Mead    0     6    12
    Powell         3    11    14
    SLSQP         16    18    18
    L-BFGS-B      15    1